# FFTW C2C Python MPI

- https://numpy.org/doc/stable/reference/routines.fft.html
- https://mpi4py-fft.readthedocs.io/en/latest/parallel.html

In [41]:
#=-----------------------------------------------------------------------=#

## Slab decomposition

In [179]:
%%writefile pfft_example.py
# https://mpi4py-fft.readthedocs.io/en/latest/parallel.html
import numpy as np, time as tm
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
t0 = tm.time()    # <--- time measurement
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
N = np.array([512, 512, 512], dtype=int)

t2 = tm.time()    # <--- time measurement
f = PFFT(comm,N,axes=(0,1,2),dtype=np.float128,grid=(-1,))
t3 = tm.time()    # <--- time measurement

u = newDistArray(f, False)
u[:] = np.random.random(u.shape).astype(u.dtype)
# Note that normalize=True is default and can be omitted
u_hat = f.forward(u, normalize=True)
uj = np.zeros_like(u)
uj = f.backward(u_hat, uj)
assert np.allclose(uj, u)
if rank == 0 :
    t1 = tm.time()    # <--- time measurement
    print('T: {:.4f} s'.format(t1-t0))
print(rank, u.shape, t3-t2)

Overwriting pfft_example.py


In [180]:
! mpiexec -n 2 python pfft_example.py

1 (256, 512, 512) 10.479369878768921
T: 34.9788 s
0 (256, 512, 512) 10.459190607070923


In [22]:
import numpy as np
np.set_printoptions(precision=2)
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
N = np.array([4, 4, 4], dtype=int)
N

array([4, 4, 4])

In [23]:
f = PFFT(MPI.COMM_WORLD,N,axes=(0,1,2),dtype=np.float128,grid=(-1,))
f

-  newDistArray() function returns a DistArray object, which in turn is a subclassed Numpy ndarray.
- uses fft to determine the size and type of the created distributed array, i.e., (64, 128, 128) and np.float for the processors.
- The False argument indicates that the shape and type should be that of the input array, as opposed to the output array type

In [24]:
u = newDistArray(f, False)
u

DistArray([[[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]]], dtype=float128)

In [27]:
u[:] = np.random.random(u.shape).astype(u.dtype)
u

DistArray([[[0.84, 0.01, 0.42, 0.78],
            [0.84, 0.09, 0.62, 0.08],
            [0.9 , 0.28, 0.73, 0.52],
            [0.41, 0.  , 0.16, 0.6 ]],

           [[0.89, 0.7 , 0.47, 0.1 ],
            [0.6 , 0.07, 0.99, 0.26],
            [0.03, 0.91, 0.55, 0.33],
            [0.23, 0.08, 0.32, 0.27]],

           [[0.43, 0.37, 0.54, 0.82],
            [0.44, 0.1 , 0.18, 0.54],
            [0.45, 0.08, 0.8 , 0.77],
            [0.75, 0.26, 0.04, 0.71]],

           [[0.32, 0.49, 0.53, 0.52],
            [0.51, 0.69, 0.67, 0.79],
            [0.46, 0.47, 0.63, 0.16],
            [0.62, 0.46, 0.33, 0.22]]], dtype=float128)

In [32]:
u_hat = f.forward(u, normalize=True)
u_hat

array([[[ 4.57e-01+0.j  ,  1.15e-02+0.04j,  6.52e-02+0.j  ],
        [ 2.17e-03-0.03j,  1.77e-02+0.03j, -5.83e-03-0.03j],
        [ 5.25e-02+0.j  , -2.25e-02-0.02j, -1.24e-02+0.j  ],
        [ 2.17e-03+0.03j,  2.65e-02-0.01j, -5.83e-03+0.03j]],

       [[-4.49e-04+0.02j,  2.26e-03-0.j  ,  4.02e-02-0.02j],
        [-6.97e-03-0.02j, -1.70e-02-0.j  , -2.09e-02-0.05j],
        [ 7.43e-03-0.03j,  1.67e-03-0.01j, -5.24e-03+0.03j],
        [-7.44e-03+0.01j,  6.44e-03-0.j  ,  2.95e-02-0.01j]],

       [[-1.78e-03+0.j  ,  3.76e-02+0.08j,  1.42e-02+0.j  ],
        [-1.23e-02+0.03j, -2.11e-02-0.j  , -2.49e-02-0.02j],
        [ 3.84e-02+0.j  , -1.80e-02+0.04j,  2.56e-02+0.j  ],
        [-1.23e-02-0.03j,  8.58e-03+0.j  , -2.49e-02+0.02j]],

       [[-4.49e-04-0.02j,  1.47e-02-0.01j,  4.02e-02+0.02j],
        [-7.44e-03-0.01j, -9.39e-04+0.02j,  2.95e-02+0.01j],
        [ 7.43e-03+0.03j,  4.65e-02+0.02j, -5.24e-03-0.03j],
        [-6.97e-03+0.02j,  1.28e-02+0.04j, -2.09e-02+0.05j]]],
      dtype=comp

In [33]:
uj = np.zeros_like(u)
uj

DistArray([[[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]],

           [[0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.],
            [0., 0., 0., 0.]]], dtype=float128)

In [34]:
uj = f.backward(u_hat, uj)
uj

DistArray([[[0.84, 0.01, 0.42, 0.78],
            [0.84, 0.09, 0.62, 0.08],
            [0.9 , 0.28, 0.73, 0.52],
            [0.41, 0.  , 0.16, 0.6 ]],

           [[0.89, 0.7 , 0.47, 0.1 ],
            [0.6 , 0.07, 0.99, 0.26],
            [0.03, 0.91, 0.55, 0.33],
            [0.23, 0.08, 0.32, 0.27]],

           [[0.43, 0.37, 0.54, 0.82],
            [0.44, 0.1 , 0.18, 0.54],
            [0.45, 0.08, 0.8 , 0.77],
            [0.75, 0.26, 0.04, 0.71]],

           [[0.32, 0.49, 0.53, 0.52],
            [0.51, 0.69, 0.67, 0.79],
            [0.46, 0.47, 0.63, 0.16],
            [0.62, 0.46, 0.33, 0.22]]], dtype=float128)

In [43]:
assert np.allclose(uj, u)

In [44]:
%%writefile example02.py
import numpy as np
np.set_printoptions(precision=2)
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
N = np.array([128, 128, 128], dtype=int)
f = PFFT(MPI.COMM_WORLD,N,axes=(0,1,2),dtype=np.float128,grid=(-1,))
u = newDistArray(f, False)
u[:] = np.random.random(u.shape).astype(u.dtype)
# Note that normalize=True is default and can be omitted
u_hat = f.forward(u, normalize=False)
print(MPI.COMM_WORLD.Get_rank(), u.shape, u_hat.shape, end='  ')
print('S: {:.2f}'.format(np.sum(u_hat)))

Overwriting example02.py


In [42]:
! mpiexec -n 2 python example02.py

0 (64, 128, 128) (128, 64, 65)  S: 969494.36+240266.90j
1 (64, 128, 128) (128, 64, 65)  S: -225488.05-496137.35j


In [81]:
import numpy as np
np.set_printoptions(precision=2)
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
N = 3
NA = np.array([N, N, N], dtype=int)
f = PFFT(MPI.COMM_WORLD,NA,axes=(0,1,2),dtype=np.complex128,grid=(-1,))
u = newDistArray(f, False)
u[:] = np.fromfunction( lambda i, j, k:
    (i*N**2 + j*N  + k + 1), u.shape, dtype=u.dtype )
print('D: {:.2f}'.format( np.sum(u) ) )
u_hat = f.forward(u, normalize=False)
print(u)
print()
print(MPI.COMM_WORLD.Get_rank(), u.shape, u_hat.shape, end='  ')
print('S: {:.2f}'.format(np.sum(u_hat)))
print(u_hat)

D: 378.00+0.00j
[[[ 1.+0.j  2.+0.j  3.+0.j]
  [ 4.+0.j  5.+0.j  6.+0.j]
  [ 7.+0.j  8.+0.j  9.+0.j]]

 [[10.+0.j 11.+0.j 12.+0.j]
  [13.+0.j 14.+0.j 15.+0.j]
  [16.+0.j 17.+0.j 18.+0.j]]

 [[19.+0.j 20.+0.j 21.+0.j]
  [22.+0.j 23.+0.j 24.+0.j]
  [25.+0.j 26.+0.j 27.+0.j]]]

0 (3, 3, 3) (3, 3, 3)  S: 27.00+0.00j
[[[ 378.  +0.j    -13.5 +7.79j  -13.5 -7.79j]
  [ -40.5+23.38j    0.  +0.j      0.  +0.j  ]
  [ -40.5-23.38j    0.  +0.j      0.  +0.j  ]]

 [[-121.5+70.15j    0.  +0.j      0.  +0.j  ]
  [   0.  +0.j      0.  +0.j      0.  +0.j  ]
  [   0.  +0.j      0.  +0.j      0.  +0.j  ]]

 [[-121.5-70.15j    0.  +0.j      0.  +0.j  ]
  [   0.  +0.j      0.  +0.j      0.  +0.j  ]
  [   0.  +0.j      0.  +0.j      0.  +0.j  ]]]


In [83]:
%%writefile example03.py
import numpy as np
np.set_printoptions(precision=2)
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
N = 3
NA = np.array([N, N, N], dtype=int)
f = PFFT(MPI.COMM_WORLD,NA,axes=(0,1,2),dtype=np.complex128,grid=(-1,))
u = newDistArray(f, False)
u[:] = np.fromfunction( lambda i, j, k:
    (i*N**2 + j*N  + k + 1), u.shape, dtype=u.dtype )
u_hat = f.forward(u, normalize=False)
print(MPI.COMM_WORLD.Get_rank(), u.shape, u_hat.shape, end='  ')
print('S: {:.2f}'.format(np.sum(u_hat)))

Overwriting example03.py


In [84]:
! mpiexec -n 1 python example03.py

0 (3, 3, 3) (3, 3, 3)  S: 27.00+0.00j


In [85]:
! mpiexec -n 2 python example03.py

0 (2, 3, 3) (3, 2, 3)  S: 67.50+23.38j
1 (1, 3, 3) (3, 1, 3)  S: -40.50-23.38j


In [130]:
%%writefile example04.py
import numpy as np
np.set_printoptions(precision=2)
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray
comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()
N = 3
NA = np.array([N, N, N], dtype=int)
f = PFFT(MPI.COMM_WORLD,NA,axes=(0,1,2),dtype=np.complex128,grid=(-1,))
u = newDistArray(f, False)
u[:] = np.fromfunction( lambda i, j, k:
    (i*N**2 + j*N  + k + 1), u.shape, dtype=u.dtype )
u_hat = f.forward(u, normalize=False)
rs = np.array(0, dtype=np.complex128)
s = np.array(np.sum(u_hat), dtype=np.complex128)
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
print(rank, u.shape, u_hat.shape, end='  ')
print('SS: {:.2f}'.format(s), end='  ')
print('S: {:.2f}'.format(rs))

Overwriting example04.py


In [131]:
! mpiexec -n 1 python example04.py

0 (3, 3, 3) (3, 3, 3)  SS: 27.00+0.00j  S: 27.00+0.00j


In [132]:
! mpiexec -n 2 python example04.py

0 (2, 3, 3) (3, 2, 3)  SS: 67.50+23.38j  S: 27.00+0.00j
1 (1, 3, 3) (3, 1, 3)  SS: -40.50-23.38j  S: 0.00+0.00j


In [210]:
%%writefile example05.py
import numpy as np, time as tm
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

t0 = tm.time()    # <--- time measurement

N = 500
NA = np.array([N, N, N], dtype=int)

f = PFFT(comm, NA, dtype=np.complex128, backend='pyfftw')
t2 = tm.time()    # <--- time measurement
print('T2 ', rank, t2-t0)

u = newDistArray(f, False)
u[:] = np.fromfunction( lambda i, j, k:
       (i*N**2 + j*N  + k + 1)*1E-6, u.shape, dtype=u.dtype )
t3 = tm.time()    # <--- time measurement
print('T3 ', rank, t3-t2)

u_hat = f.forward(u, normalize=False)
t4 = tm.time()    # <--- time measurement
print('T4 ', rank, t4-t3)

rs = np.array(0, dtype=np.complex128)
s = np.array(np.sum(u_hat), dtype=np.complex128)
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
t5 = tm.time()    # <--- time measurement
print('T5 ', rank, t5-t4)

if rank == 0 :
    t1 = tm.time()    # <--- time measurement
    print('S: {:.2f}    T: {:.4f} s'.format(rs, t1-t0))

Overwriting example05.py


In [211]:
! mpiexec -n 1 python example05.py

T2  0 4.783401012420654
T3  0 9.118717193603516
T4  0 14.764372110366821
T5  0 0.14860916137695312
S: 125.00+0.00j    T: 28.8151 s


In [207]:
! mpiexec -n 2 python example05.py

1 3.0494768619537354
0 1.917353868484497
S: 125.00+0.00j    T: 14.9763 s


In [3]:
%%writefile example06.py
import numpy as np, time as tm
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
if rank == 0 :
    t0 = tm.time()    # <--- time measurement

N = 500
NA = np.array([N, N, N], dtype=int)
f = PFFT(comm, NA, dtype=np.complex128, backend='pyfftw')
u = newDistArray(f, False)
u[:] = np.fromfunction( lambda i, j, k:
       (i*N**2 + j*N  + k + 1)*1E-6, u.shape, dtype=u.dtype )

u_hat = f.forward(u, normalize=False)

rs = np.array(0, dtype=np.complex128)
s = np.array(np.sum(u_hat), dtype=np.complex128)
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
if rank == 0 :
    t1 = tm.time()    # <--- time measurement
    print('S: {:.2f}    T: {:.4f} s'.format(rs, t1-t0))

Writing example06.py


- module load  anaconda3/2020.11
- module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
- conda activate /scratch/app/anaconda3/2020.11
- conda activate $pwd/env2

In [1]:
! module list

Currently Loaded Modulefiles:
  1) anaconda3/2020.11
  2) openmpi/gnu/3.1.4
  3) mathlibs/fftw/3.3.8_openmpi-3.1_gnu


In [10]:
! mpiexec -n 1 python example06.py

S: 125.00+0.00j    T: 41.4103 s


In [5]:
! mpiexec -n 2 python example06.py

S: 125.00+0.00j    T: 20.4675 s


In [7]:
! mpiexec -n 4 python example06.py

S: 125.00-0.00j    T: 10.5906 s


In [8]:
! mpiexec -n 8 python example06.py

S: 125.00-0.00j    T: 6.0011 s


In [9]:
! mpiexec -n 16 python example06.py

S: 125.00-0.00j    T: 4.5152 s


## Runs on an execution node

Rename and copy to /scratch

In [1]:
%%bash
mv example06.py pc2cp.py
dst=/scratch${PWD#"/prj"}
mkdir -p $dst
cp pc2cp.py $dst

## Batch script

In [33]:
%%writefile pc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name pc2cp       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes
#   NTASKS: 1, 4, 9, 16, 36, 49, 64, 81
echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd
cd /scratch${PWD#"/prj"}/    # maybe wrong, need check
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate --stack ./envs
cd fft
                                              
# Executable
EXEC="python pc2cp.py"

# Start
echo '$ srun --mpi=pmi2 -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun --mpi=pmi2 -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting pc2cp.srm


Check

In [12]:
%%bash
cd
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate --stack ./envs
mpiexec -n 20 python fft/pc2cp.py

S: 125.00+0.00j    T: 1.9114 s


In [38]:
! sbatch --partition=cpu_dev --ntasks=9 pc2cp.srm

Submitted batch job 868368


In [39]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [40]:
! cat /scratch${PWD#"/prj"}/slurm-868368.out

- Job ID: 868368
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1244
$ srun -n 9 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.3468 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Submit batch script

In [3]:
! sbatch --partition=cpu_dev pc2cp.srm

Submitted batch job 864643


In [4]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864643    cpu_dev   R  0:00     1   24


In [9]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [10]:
! cat /scratch${PWD#"/prj"}/slurm-864643.out

- Job ID: 864643
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1247
$ srun -n 1 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 22.5614 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [41]:
! sbatch --ntasks=1 pc2cp.srm
! sbatch --ntasks=1 pc2cp.srm
! sbatch --ntasks=1 pc2cp.srm

Submitted batch job 868369
Submitted batch job 868370
Submitted batch job 868371


In [42]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868369  cpu_small  PD  0:00     1    1
            868370  cpu_small  PD  0:00     1    1
            868371  cpu_small  PD  0:00     1    1


In [ ]:
! cat /scratch${PWD#"/prj"}/slurm-868369.out
! cat /scratch${PWD#"/prj"}/slurm-868369.out
! cat /scratch${PWD#"/prj"}/slurm-868371.out

- Job ID: 864668
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
$ srun -n 1 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 25.0917 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 864669
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
$ srun -n 1 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 22.9735 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 864670
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
$ srun -n 1 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 24.7515 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [43]:
! sbatch --ntasks=4 pc2cp.srm
! sbatch --ntasks=4 pc2cp.srm
! sbatch --ntasks=4 pc2cp.srm

Submitted batch job 868372
Submitted batch job 868373
Submitted batch job 868374


In [44]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868371  cpu_small  PD  0:00     1    1
            868370  cpu_small  PD  0:00     1    1
            868369  cpu_small  PD  0:00     1    1
            868372  cpu_small  PD  0:00     1    4
            868373  cpu_small  PD  0:00     1    4
            868374  cpu_small  PD  0:00     1    4


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868372.out
! cat /scratch${PWD#"/prj"}/slurm-868373.out
! cat /scratch${PWD#"/prj"}/slurm-868374.out

- Job ID: 868372
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 4 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.8820 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868373
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 4 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.8020 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868374
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 4 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.7180 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [45]:
! sbatch --ntasks=9 pc2cp.srm
! sbatch --ntasks=9 pc2cp.srm
! sbatch --ntasks=9 pc2cp.srm

Submitted batch job 868375
Submitted batch job 868376
Submitted batch job 868377


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868375.out
! cat /scratch${PWD#"/prj"}/slurm-868376.out
! cat /scratch${PWD#"/prj"}/slurm-868377.out

- Job ID: 868375
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 9 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.3969 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868376
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 9 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.3931 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868377
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 9 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.4361 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [46]:
! sbatch --ntasks=16 pc2cp.srm
! sbatch --ntasks=16 pc2cp.srm
! sbatch --ntasks=16 pc2cp.srm

Submitted batch job 868379
Submitted batch job 868380
Submitted batch job 868381


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-868379.out
! cat /scratch${PWD#"/prj"}/slurm-868380.out
! cat /scratch${PWD#"/prj"}/slurm-868381.out

- Job ID: 868379
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 16 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.0623 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868380
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 16 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.0992 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868381
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
$ srun -n 16 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.0754 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [47]:
! sbatch --ntasks=36 pc2cp.srm
! sbatch --ntasks=36 pc2cp.srm
! sbatch --ntasks=36 pc2cp.srm

Submitted batch job 868382
Submitted batch job 868383
Submitted batch job 868384


In [5]:
! cat /scratch${PWD#"/prj"}/slurm-868382.out
! cat /scratch${PWD#"/prj"}/slurm-868383.out
! cat /scratch${PWD#"/prj"}/slurm-868384.out

- Job ID: 868382
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun -n 36 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.8288 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868383
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun -n 36 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.2874 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868384
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
$ srun -n 36 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 7.2990 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [48]:
! sbatch --ntasks=49 pc2cp.srm
! sbatch --ntasks=49 pc2cp.srm
! sbatch --ntasks=49 pc2cp.srm

Submitted batch job 868385
Submitted batch job 868386
Submitted batch job 868387


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868385.out
! cat /scratch${PWD#"/prj"}/slurm-868386.out
! cat /scratch${PWD#"/prj"}/slurm-868387.out

- Job ID: 868385
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1337
$ srun -n 49 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 7.6719 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868386
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun -n 49 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.4202 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868387
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun -n 49 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.5554 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [49]:
! sbatch --ntasks=64 pc2cp.srm
! sbatch --ntasks=64 pc2cp.srm
! sbatch --ntasks=64 pc2cp.srm

Submitted batch job 868388
Submitted batch job 868389
Submitted batch job 868390


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868388.out
! cat /scratch${PWD#"/prj"}/slurm-868389.out
! cat /scratch${PWD#"/prj"}/slurm-868390.out

- Job ID: 868388
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
$ srun -n 64 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 6.0906 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868389
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
$ srun -n 64 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 6.3464 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868390
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
$ srun -n 64 python pc2cp.py
-- output -----------------------------
S: 125.00+0.00j    T: 6.1094 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [55]:
! sbatch --ntasks=81 pc2cp.srm
! sbatch --ntasks=81 pc2cp.srm
! sbatch --ntasks=81 pc2cp.srm

Submitted batch job 868395
Submitted batch job 868396
Submitted batch job 868397


In [56]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868371  cpu_small  PD  0:00     1    1
            868370  cpu_small  PD  0:00     1    1
            868369  cpu_small  PD  0:00     1    1
            868374  cpu_small  PD  0:00     1    4
            868373  cpu_small  PD  0:00     1    4
            868372  cpu_small  PD  0:00     1    4
            868381  cpu_small  PD  0:00     1   16
            868380  cpu_small  PD  0:00     1   16
            868379  cpu_small  PD  0:00     1   16
            868377  cpu_small  PD  0:00     1    9
            868376  cpu_small  PD  0:00     1    9
            868375  cpu_small  PD  0:00     1    9
            868382  cpu_small  PD  0:00     2   36
            868383  cpu_small  PD  0:00     2   36
            868384  cpu_small  PD  0:00     2   36
            868385  cpu_small  PD  0:00     3   49
            868386  cpu_small  PD  0:00     3   49
            868387  cpu_small  PD  0:00     3   49
            868388  cpu_small  

In [58]:
! squeue -u $(whoami) -h -t pending,running -r | wc -l

48


In [1]:
! squeue -n pc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-868395.out
! cat /scratch${PWD#"/prj"}/slurm-868396.out
! cat /scratch${PWD#"/prj"}/slurm-868397.out

- Job ID: 868395
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
$ srun -n 81 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 5.7121 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868396
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1337 sdumont1338
$ srun -n 81 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 5.9182 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868397
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
$ srun -n 81 python pc2cp.py
-- output -----------------------------
S: 125.00-0.00j    T: 5.2087 s
~~